# Day 15: Building RAG with LangChain & Chroma

Welcome to Day 15! Today we're building a complete RAG (Retrieval Augmented Generation) system.

## What We'll Build Today:

1. **Document Chunking** - Break documents into manageable pieces
2. **Vector Embeddings** - Convert text to numerical representations
3. **Vector Database** - Store and search vectors with Chroma
4. **Visualization** - See vectors in 2D and 3D space
5. **RAG Pipeline** - Complete retrieval and generation system
6. **Gradio App** - User-friendly chat interface

Let's get started!


## Setup: Import Libraries and Load Environment

First, let's import all the libraries we'll need and load our OpenAI API key from the `.env` file.


In [1]:
# Import necessary libraries
import os
from dotenv import load_dotenv
import glob
import shutil

# LangChain imports
from langchain_community.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.schema import SystemMessage, HumanMessage, AIMessage

# Visualization imports
import plotly.express as px
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import numpy as np

# Gradio for UI
import gradio as gr

# Load environment variables
load_dotenv()

# Verify OpenAI API key is loaded
api_key = os.getenv("OPENAI_API_KEY")
if api_key:
    print("✅ OpenAI API key loaded successfully!")
else:
    print("❌ OpenAI API key not found. Please check your .env file.")

# Set model name
MODEL_NAME = "gpt-4o-mini"
print(f"Using model: {MODEL_NAME}")


✅ OpenAI API key loaded successfully!
Using model: gpt-4o-mini


## Part 1: Create Sample Knowledge Base

Let's create a simple knowledge base with sample documents about a fictional insurance company called "InsureElm".


In [2]:
# Create knowledge base directory structure
os.makedirs("knowledge_base/employees", exist_ok=True)
os.makedirs("knowledge_base/products", exist_ok=True)
os.makedirs("knowledge_base/contracts", exist_ok=True)
os.makedirs("knowledge_base/company", exist_ok=True)

# Sample employee document
employee_doc = """# Employee Profile: Avery Lancaster

**Name:** Avery Lancaster
**Position:** Co-founder and Chief Executive Officer (CEO)
**Department:** Executive Leadership
**Employee ID:** EMP-001
**Start Date:** January 15, 2020

## Background
Avery Lancaster is the Co-founder and CEO of InsureElm, an innovative insurance technology company. She has extensive experience in the insurance and technology sectors, with over 15 years of leadership experience.

## Education
- MBA from Stanford Graduate School of Business
- BS in Computer Science from MIT

## Previous Experience
Before founding InsureElm, Avery served as Vice President of Product at TechCorp, where she led the development of enterprise software solutions. She also worked as a Senior Product Manager at FinanceInnovate.

## Key Achievements
- Led InsureElm from startup to $50M in annual revenue
- Pioneered AI-powered claims processing in the insurance industry
- Named "Top 40 Under 40" by Insurance Business Magazine

## Contact
Email: avery.lancaster@insureelm.com
Phone: (555) 123-4567
"""

# Sample product document
product_doc = """# Product: CarElm Auto Insurance

## Overview
CarElm is InsureElm's flagship auto insurance product, designed to provide comprehensive coverage with competitive rates and exceptional customer service.

## Features
- **Comprehensive Coverage:** Collision, liability, and comprehensive protection
- **Flexible Deductibles:** Choose from $250, $500, or $1,000 deductibles
- **Accident Forgiveness:** First accident won't increase your premium
- **24/7 Roadside Assistance:** Towing, jump-starts, and emergency fuel delivery
- **Mobile App:** File claims, view policy details, and get instant support

## Pricing
Starting at just $89/month for basic coverage. Premium plans available from $149/month.

## Coverage Options
- Bodily Injury Liability: Up to $500,000 per accident
- Property Damage Liability: Up to $250,000 per accident
- Collision Coverage: Actual cash value minus deductible
- Comprehensive Coverage: Protection against theft, vandalism, and natural disasters
- Uninsured Motorist Coverage: Protection when others don't have insurance

## Claims Process
Our AI-powered ClaimElm system processes most claims in under 24 hours. Simply:
1. Report the incident through our mobile app
2. Upload photos and documentation
3. Receive instant claim approval for eligible claims
4. Get paid directly to your bank account

## Customer Satisfaction
- 4.8/5 star rating on TrustPilot
- 95% customer retention rate
- Average claim processing time: 18 hours
"""

# Sample contract document
contract_doc = """# Service Agreement Contract

## InsureElm Service Level Agreement

**Contract Number:** SLA-2024-001
**Effective Date:** January 1, 2024
**Term:** 12 months with automatic renewal

## Service Commitments

### Claims Processing
InsureElm commits to processing standard claims within 24 hours of submission. Complex claims will be processed within 5 business days.

### Customer Support
- 24/7 phone support with average wait time under 2 minutes
- Email response within 4 hours during business hours
- Live chat support available 8 AM - 10 PM EST

### System Availability
Our digital platforms will maintain 99.9% uptime, excluding scheduled maintenance windows.

### Data Security
All customer data is encrypted using AES-256 encryption. We comply with:
- GDPR (General Data Protection Regulation)
- CCPA (California Consumer Privacy Act)
- SOC 2 Type II certification

## Integration Services
We provide API access for partners and enterprise clients with:
- RESTful API endpoints
- Webhook notifications for claim status updates
- Real-time policy management
- Automated billing integration

## Performance Metrics
- Claim approval rate: 92%
- Customer satisfaction score: 4.7/5
- Average policy renewal rate: 88%

## Termination Clause
Either party may terminate this agreement with 30 days written notice. All active policies will be honored through their term.
"""

# Sample company document
company_doc = """# About InsureElm

## Company Overview
InsureElm is a technology-driven insurance company founded in 2020 by Avery Lancaster and Jordan Rivera. We combine cutting-edge AI technology with traditional insurance expertise to provide better coverage at lower costs.

## Mission Statement
To make insurance simple, transparent, and accessible for everyone through innovative technology and exceptional customer service.

## Company Values
1. **Innovation:** We constantly seek better ways to serve our customers
2. **Transparency:** No hidden fees, clear policies, straightforward communication
3. **Customer First:** Every decision is made with our customers' best interests in mind
4. **Integrity:** We do what's right, even when no one is watching

## Products
- **CarElm:** Auto insurance with AI-powered claims
- **HealthElm:** Comprehensive health insurance plans
- **LifeElm:** Term and whole life insurance
- **HomeElm:** Homeowners and renters insurance

## Technology Platform
Our proprietary ClaimElm AI system uses machine learning to:
- Detect fraud automatically
- Process claims in under 24 hours
- Provide instant quotes
- Personalize coverage recommendations

## Company Statistics
- Founded: 2020
- Headquarters: San Francisco, CA
- Employees: 250+
- Customers: 50,000+
- Annual Revenue: $50M+
- States Covered: 45

## Leadership Team
- **Avery Lancaster:** CEO and Co-founder
- **Jordan Rivera:** CTO and Co-founder
- **Maxine Thompson:** Senior Data Engineer
- **Dr. Sarah Chen:** Chief Medical Officer
- **Marcus Williams:** VP of Claims

## Contact Information
**Headquarters:**
123 Innovation Drive
San Francisco, CA 94105

**Phone:** 1-800-INSURE-ELM
**Email:** info@insureelm.com
**Website:** www.insureelm.com
"""

# Write documents to files
with open("knowledge_base/employees/avery_lancaster.md", "w") as f:
    f.write(employee_doc)

with open("knowledge_base/products/car_insurance.md", "w") as f:
    f.write(product_doc)

with open("knowledge_base/contracts/service_agreement.md", "w") as f:
    f.write(contract_doc)

with open("knowledge_base/company/about.md", "w") as f:
    f.write(company_doc)

print("✅ Knowledge base created with 4 sample documents")
print("   - 1 employee profile")
print("   - 1 product description")
print("   - 1 contract")
print("   - 1 company overview")


✅ Knowledge base created with 4 sample documents
   - 1 employee profile
   - 1 product description
   - 1 contract
   - 1 company overview


## Part 2: Load Documents with LangChain

Now let's use LangChain's document loaders to load all our markdown files.


In [3]:
# Get list of all folders in knowledge base
folders = glob.glob("knowledge_base/*/")
# Use os.path.basename to work on both Windows and Unix
print(f"Found {len(folders)} folders: {[os.path.basename(os.path.normpath(f)) for f in folders]}")

# Load documents from all folders
documents = []

for folder in folders:
    # Get folder name (e.g., "employees", "products")
    # Use os.path.basename to work cross-platform (Windows uses backslash)
    doc_type = os.path.basename(os.path.normpath(folder))
    
    # Find all markdown files in this folder
    md_files = glob.glob(os.path.join(folder, "*.md"))
    
    # Load each markdown file
    for md_file in md_files:
        # Read the file content
        with open(md_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Create a LangChain Document object
        from langchain.schema import Document
        doc = Document(
            page_content=content,
            metadata={
                "source": md_file,
                "doc_type": doc_type
            }
        )
        documents.append(doc)
    
    print(f"  Loaded {len(md_files)} documents from {doc_type}/")

print(f"\n✅ Total documents loaded: {len(documents)}")

# Let's examine one document
print("\n--- Sample Document ---")
print(f"Source: {documents[0].metadata['source']}")
print(f"Type: {documents[0].metadata['doc_type']}")
print(f"Content preview: {documents[0].page_content[:200]}...")


Found 4 folders: ['company', 'contracts', 'employees', 'products']
  Loaded 1 documents from company/
  Loaded 1 documents from contracts/
  Loaded 1 documents from employees/
  Loaded 1 documents from products/

✅ Total documents loaded: 4

--- Sample Document ---
Source: knowledge_base\company\about.md
Type: company
Content preview: # About InsureElm

## Company Overview
InsureElm is a technology-driven insurance company founded in 2020 by Avery Lancaster and Jordan Rivera. We combine cutting-edge AI technology with traditional i...


## Part 3: Document Chunking

Documents are often too large to process as a whole. We'll split them into smaller chunks that are more focused and easier to match with user queries.

**Why chunk?**
- User questions are specific
- Chunks provide focused context
- Better matching between query and content
- More efficient retrieval


In [4]:
# Create text splitter
# RecursiveCharacterTextSplitter tries to split at natural boundaries:
# 1. Double newlines (paragraphs)
# 2. Single newlines (lines)
# 3. Sentences
# 4. Words

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Target chunk size in characters
    chunk_overlap=200,    # Overlap to prevent splitting answers
    length_function=len,
    is_separator_regex=False,
)

# Split documents into chunks
chunks = text_splitter.split_documents(documents)

print(f"✅ Split {len(documents)} documents into {len(chunks)} chunks")
print(f"   Average chunks per document: {len(chunks) / len(documents):.1f}")

# Examine a chunk
print("\n--- Sample Chunk ---")
print(f"Source: {chunks[0].metadata['source']}")
print(f"Type: {chunks[0].metadata['doc_type']}")
print(f"Length: {len(chunks[0].page_content)} characters")
print(f"\nContent:\n{chunks[0].page_content}")


✅ Split 4 documents into 8 chunks
   Average chunks per document: 2.0

--- Sample Chunk ---
Source: knowledge_base\company\about.md
Type: company
Length: 957 characters

Content:
# About InsureElm

## Company Overview
InsureElm is a technology-driven insurance company founded in 2020 by Avery Lancaster and Jordan Rivera. We combine cutting-edge AI technology with traditional insurance expertise to provide better coverage at lower costs.

## Mission Statement
To make insurance simple, transparent, and accessible for everyone through innovative technology and exceptional customer service.

## Company Values
1. **Innovation:** We constantly seek better ways to serve our customers
2. **Transparency:** No hidden fees, clear policies, straightforward communication
3. **Customer First:** Every decision is made with our customers' best interests in mind
4. **Integrity:** We do what's right, even when no one is watching

## Products
- **CarElm:** Auto insurance with AI-powered claims
- **Health

## Part 4: Create Vector Embeddings

Now we'll convert our text chunks into vector embeddings using an encoder model. We'll start with the open-source **all-MiniLM-L6-v2** model from Hugging Face.

**Key Concept:** 
- **Encoder Model** creates the vectors (affects quality)
- **Vector Database** stores the vectors (affects speed)
- They are separate things!


In [5]:
# Create embedding model
# This will download the model on first run (~100MB)
print("Creating embedding model (this may take a moment on first run)...")

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded!")
print(f"   Model: all-MiniLM-L6-v2")
print(f"   Dimensions: 384")
print(f"   Cost: Free (open source)")

# Test the embedding model
test_text = "Avery Lancaster is the CEO of InsureElm"
test_embedding = embeddings.embed_query(test_text)

print(f"\n--- Test Embedding ---")
print(f"Text: '{test_text}'")
print(f"Vector dimensions: {len(test_embedding)}")
print(f"First 10 values: {test_embedding[:10]}")


Creating embedding model (this may take a moment on first run)...
✅ Embedding model loaded!
   Model: all-MiniLM-L6-v2
   Dimensions: 384
   Cost: Free (open source)

--- Test Embedding ---
Text: 'Avery Lancaster is the CEO of InsureElm'
Vector dimensions: 384
First 10 values: [-0.06184487044811249, 0.03463773429393768, 0.018159912899136543, -0.0008979347185231745, 0.010728622786700726, -0.01033355575054884, 0.08555151522159576, 0.011272937059402466, -0.022408023476600647, -0.012546326965093613]


## Part 5: Create Chroma Vector Database

Now we'll create a Chroma vector database to store our chunks as vectors.


In [6]:
# Directory for vector database
vector_db_path = "./vector_db"

# Close any existing vector store connection
try:
    if 'vector_store' in globals():
        del vector_store
    import gc
    gc.collect()  # Force garbage collection to release file handles
except:
    pass

# Delete existing database if it exists (for clean start)
if os.path.exists(vector_db_path):
    import time
    time.sleep(0.5)  # Give Windows time to release file handles
    try:
        shutil.rmtree(vector_db_path)
        print("Deleted existing vector database")
    except PermissionError:
        print("⚠️  Could not delete existing database (files in use)")
        print("   Using existing database instead")

# Create Chroma vector store
print(f"\nCreating Chroma vector store with {len(chunks)} chunks...")
print("This will take a moment as each chunk is converted to a vector...")

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=vector_db_path
)

print(f"\n✅ Vector store created successfully!")

# Inspect the vector store
collection = vector_store._collection
count = collection.count()
sample = collection.get(limit=1, include=["embeddings"])

# Check if embeddings exist and get dimensions
try:
    if sample["embeddings"] is not None and len(sample["embeddings"]) > 0:
        dimensions = len(sample["embeddings"][0])
    else:
        dimensions = 0
except (KeyError, TypeError, IndexError):
    dimensions = 0

print(f"   Total vectors: {count}")
print(f"   Vector dimensions: {dimensions}")
print(f"   Storage location: {vector_db_path}")


Deleted existing vector database

Creating Chroma vector store with 8 chunks...
This will take a moment as each chunk is converted to a vector...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given



✅ Vector store created successfully!
   Total vectors: 8
   Vector dimensions: 384
   Storage location: ./vector_db


In [7]:
# Get list of all folders in knowledge base
folders = glob.glob("knowledge_base/*/")
# Use os.path.basename to work on both Windows and Unix
print(f"Found {len(folders)} folders: {[os.path.basename(os.path.normpath(f)) for f in folders]}")

# Load documents from all folders
documents = []

for folder in folders:
    # Get folder name (e.g., "employees", "products")
    # Use os.path.basename to work cross-platform (Windows uses backslash)
    doc_type = os.path.basename(os.path.normpath(folder))
    
    # Find all markdown files in this folder
    md_files = glob.glob(os.path.join(folder, "*.md"))
    
    # Load each markdown file
    for md_file in md_files:
        # Read the file content
        with open(md_file, 'r', encoding='utf-8') as f:
            content = f.read()
        
        # Create a LangChain Document object
        from langchain.schema import Document
        doc = Document(
            page_content=content,
            metadata={
                "source": md_file,
                "doc_type": doc_type
            }
        )
        documents.append(doc)
    
    print(f"  Loaded {len(md_files)} documents from {doc_type}/")

print(f"\n✅ Total documents loaded: {len(documents)}")

# Let's examine one document
print("\n--- Sample Document ---")
print(f"Source: {documents[0].metadata['source']}")
print(f"Type: {documents[0].metadata['doc_type']}")
print(f"Content preview: {documents[0].page_content[:200]}...")


Found 4 folders: ['company', 'contracts', 'employees', 'products']
  Loaded 1 documents from company/
  Loaded 1 documents from contracts/
  Loaded 1 documents from employees/
  Loaded 1 documents from products/

✅ Total documents loaded: 4

--- Sample Document ---
Source: knowledge_base\company\about.md
Type: company
Content preview: # About InsureElm

## Company Overview
InsureElm is a technology-driven insurance company founded in 2020 by Avery Lancaster and Jordan Rivera. We combine cutting-edge AI technology with traditional i...


## Part 6: Visualize Vectors with t-SNE

Let's visualize our high-dimensional vectors in 2D and 3D space using t-SNE (t-distributed Stochastic Neighbor Embedding).

**t-SNE** projects high-dimensional data to lower dimensions while preserving similarity relationships.


In [8]:
# Get all vectors and metadata from Chroma
collection = vector_store._collection
data = collection.get(include=["embeddings", "documents", "metadatas"])

vectors = np.array(data["embeddings"])
texts = data["documents"]
metadatas = data["metadatas"]

# Extract document types for coloring
doc_types = [meta.get("doc_type", "unknown") for meta in metadatas]

print(f"Retrieved {len(vectors)} vectors")
print(f"Vector shape: {vectors.shape}")
print(f"Document types: {set(doc_types)}")


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Retrieved 8 vectors
Vector shape: (8, 384)
Document types: {'contracts', 'company', 'employees', 'products'}


### 2D Visualization


In [9]:
# Apply t-SNE to reduce to 2 dimensions
print("Applying t-SNE (2D)... this may take a moment")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
vectors_2d = tsne_2d.fit_transform(vectors)

# Create interactive 2D scatter plot
fig_2d = px.scatter(
    x=vectors_2d[:, 0],
    y=vectors_2d[:, 1],
    color=doc_types,
    hover_data={"text": [text[:100] + "..." for text in texts]},
    title="2D Visualization of Document Vectors (t-SNE)",
    labels={"x": "t-SNE Dimension 1", "y": "t-SNE Dimension 2", "color": "Document Type"},
    width=900,
    height=600
)

fig_2d.update_traces(marker=dict(size=8, opacity=0.7))
fig_2d.show()

print("\n✅ 2D visualization complete!")
print("   Hover over points to see text content")
print("   Similar documents should cluster together")


Applying t-SNE (2D)... this may take a moment



✅ 2D visualization complete!
   Hover over points to see text content
   Similar documents should cluster together


### 3D Visualization


In [10]:
# Apply t-SNE to reduce to 3 dimensions
print("Applying t-SNE (3D)... this may take a moment")
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(30, len(vectors)-1))
vectors_3d = tsne_3d.fit_transform(vectors)

# Create interactive 3D scatter plot
fig_3d = px.scatter_3d(
    x=vectors_3d[:, 0],
    y=vectors_3d[:, 1],
    z=vectors_3d[:, 2],
    color=doc_types,
    hover_data={"text": [text[:100] + "..." for text in texts]},
    title="3D Visualization of Document Vectors (t-SNE)",
    labels={"x": "Dimension 1", "y": "Dimension 2", "z": "Dimension 3", "color": "Document Type"},
    width=900,
    height=700
)

fig_3d.update_traces(marker=dict(size=5, opacity=0.7))
fig_3d.show()

print("\n✅ 3D visualization complete!")
print("   Rotate the plot to explore from different angles")
print("   Zoom in to see individual clusters")


Applying t-SNE (3D)... this may take a moment



✅ 3D visualization complete!
   Rotate the plot to explore from different angles
   Zoom in to see individual clusters


## Part 7: Build RAG Pipeline

Now let's build the complete RAG (Retrieval Augmented Generation) system!

**RAG Flow:**
1. User asks a question
2. Convert question to vector
3. Find similar vectors in database
4. Retrieve the text chunks
5. Add chunks to prompt as context
6. LLM generates answer using context


In [11]:
# Create retriever from vector store
# k=5 means retrieve the top 5 most similar chunks
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

# Create LLM
llm = ChatOpenAI(model=MODEL_NAME, temperature=0)

print("✅ RAG components ready!")
print(f"   Retriever: Will fetch top 5 similar chunks")
print(f"   LLM: {MODEL_NAME}")


✅ RAG components ready!
   Retriever: Will fetch top 5 similar chunks
   LLM: gpt-4o-mini


### Simple RAG Function


In [12]:
def answer_question_simple(question):
    """
    Simple RAG function that answers a question using retrieved context.
    
    Args:
        question: User's question as a string
        
    Returns:
        Answer as a string
    """
    # Step 1: Retrieve relevant context
    context_docs = retriever.invoke(question)
    
    # Step 2: Combine context into a single string
    context = "\n\n".join([doc.page_content for doc in context_docs])
    
    # Step 3: Create system prompt with context
    system_prompt = f"""You are an expert assistant for InsureElm insurance company.
Use the following context to answer the user's question accurately and concisely.
If the answer is not in the context, say so.

Context:
{context}

Answer the question based on the context provided."""
    
    # Step 4: Build messages
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]
    
    # Step 5: Get response from LLM
    response = llm.invoke(messages)
    
    return response.content

# Test the function
print("Testing RAG system...\n")

test_question = "Who is Avery Lancaster?"
answer = answer_question_simple(test_question)

print(f"Question: {test_question}")
print(f"Answer: {answer}")


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Testing RAG system...

Question: Who is Avery Lancaster?
Answer: Avery Lancaster is the Co-founder and Chief Executive Officer (CEO) of InsureElm, an innovative insurance technology company. She has over 15 years of leadership experience in the insurance and technology sectors. Avery holds an MBA from Stanford Graduate School of Business and a BS in Computer Science from MIT. Before founding InsureElm, she served as Vice President of Product at TechCorp and worked as a Senior Product Manager at FinanceInnovate. Under her leadership, InsureElm has grown to $50M in annual revenue and pioneered AI-powered claims processing in the insurance industry. She has also been recognized as one of the "Top 40 Under 40" by Insurance Business Magazine.


In [13]:
# Test with more questions
test_questions = [
    "What is CarElm?",
    "What products does InsureElm offer?",
    "Tell me about the claims processing time",
    "Who is the CEO?",  # Different phrasing
    "Who is Avry Lancster?",  # Typo - fuzzy matching!
]

print("\n" + "="*60)
print("Testing RAG with Multiple Questions")
print("="*60 + "\n")

for q in test_questions:
    answer = answer_question_simple(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("-" * 60 + "\n")



Testing RAG with Multiple Questions

Q: What is CarElm?
A: CarElm is InsureElm's flagship auto insurance product, designed to provide comprehensive coverage with competitive rates and exceptional customer service. It includes features such as collision, liability, and comprehensive protection, flexible deductibles, accident forgiveness, 24/7 roadside assistance, and a mobile app for managing claims and policy details.

------------------------------------------------------------

Q: What products does InsureElm offer?
A: InsureElm offers the following products:

1. **CarElm:** Auto insurance with AI-powered claims
2. **HealthElm:** Comprehensive health insurance plans
3. **LifeElm:** Term and whole life insurance
4. **HomeElm:** Homeowners and renters insurance

------------------------------------------------------------

Q: Tell me about the claims processing time
A: InsureElm commits to processing standard claims within 24 hours of submission. Complex claims will be processed withi

### RAG with Conversation History

For a chatbot, we need to handle conversation history since LLMs are stateless.


In [14]:
def answer_question_with_history(question, history=[]):
    """
    RAG function that handles conversation history.
    
    Args:
        question: Current user question
        history: List of tuples [(user_msg, ai_msg), ...]
        
    Returns:
        Answer as a string
    """
    # For better context retrieval, combine all user questions
    # This helps with follow-up questions like "What did she do before?"
    all_user_questions = [q for q, a in history] + [question]
    combined_question = " ".join(all_user_questions)
    
    # Retrieve context based on combined question
    context_docs = retriever.invoke(combined_question)
    context = "\n\n".join([doc.page_content for doc in context_docs])
    
    # Create system prompt
    system_prompt = f"""You are an expert assistant for InsureElm insurance company.
Use the following context to answer questions accurately.

Context:
{context}

Answer based on the context and conversation history."""
    
    # Build messages with full history
    messages = [SystemMessage(content=system_prompt)]
    
    # Add conversation history
    for user_msg, ai_msg in history:
        messages.append(HumanMessage(content=user_msg))
        messages.append(AIMessage(content=ai_msg))
    
    # Add current question
    messages.append(HumanMessage(content=question))
    
    # Get response
    response = llm.invoke(messages)
    
    return response.content

# Test with conversation history
print("Testing conversation with follow-up questions...\n")

# First question
q1 = "Who is Avery?"
a1 = answer_question_with_history(q1, [])
print(f"User: {q1}")
print(f"AI: {a1}\n")

# Follow-up question (uses "she" - needs history!)
history = [(q1, a1)]
q2 = "What did she do before InsureElm?"
a2 = answer_question_with_history(q2, history)
print(f"User: {q2}")
print(f"AI: {a2}\n")

# Another follow-up
history.append((q2, a2))
q3 = "What is her education background?"
a3 = answer_question_with_history(q3, history)
print(f"User: {q3}")
print(f"AI: {a3}")


Testing conversation with follow-up questions...

User: Who is Avery?
AI: Avery Lancaster is the Co-founder and Chief Executive Officer (CEO) of InsureElm, an innovative insurance technology company. She has over 15 years of leadership experience in the insurance and technology sectors. Avery holds an MBA from Stanford Graduate School of Business and a BS in Computer Science from MIT. She has previously served as Vice President of Product at TechCorp and as a Senior Product Manager at FinanceInnovate. Under her leadership, InsureElm has grown from a startup to achieving $50 million in annual revenue and has pioneered AI-powered claims processing in the insurance industry.

User: What did she do before InsureElm?
AI: Before founding InsureElm, Avery Lancaster served as Vice President of Product at TechCorp, where she led the development of enterprise software solutions. She also worked as a Senior Product Manager at FinanceInnovate.

User: What is her education background?
AI: Avery Lan

## Part 8: Gradio Chat Interface

Now let's create a user-friendly chat interface using Gradio.


In [16]:
# Create Gradio chat interface
# Note: Gradio 5.x has a simpler API - removed retry_btn, undo_btn, clear_btn parameters
demo = gr.ChatInterface(
    fn=answer_question_with_history,
    title="InsureElm RAG Assistant",
    description="Ask questions about InsureElm employees, products, and services. Powered by RAG!",
    examples=[
        "Who is Avery Lancaster?",
        "What is CarElm?",
        "What products does InsureElm offer?",
        "Tell me about the claims processing",
        "Who is the CEO?",
    ],
    theme="soft"
)

# Launch the interface
print("Launching Gradio interface...")
print("The interface will open in a new browser tab")
print("Press Ctrl+C to stop the server\n")

demo.launch(share=False)


c:\Users\HP\.conda\envs\llm-env\Lib\site-packages\gradio\chat_interface.py:339: UserWarning:

The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.



Launching Gradio interface...
The interface will open in a new browser tab
Press Ctrl+C to stop the server

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Part 9: Comparing Different Encoders (Optional)

Let's see how different encoder models affect the quality of our vector space. We'll compare:
1. **all-MiniLM-L6-v2** (384 dims) - Free, open source
2. **text-embedding-3-small** (1,536 dims) - OpenAI, paid
3. **text-embedding-3-large** (3,072 dims) - OpenAI, best quality


In [19]:
# Function to create vector store with different encoders
def create_vector_store_with_encoder(encoder_name, encoder_obj, chunks):
    """Create a vector store with a specific encoder"""
    db_path = f"./vector_db_{encoder_name}"
    
    # Delete if exists (with Windows-friendly error handling)
    if os.path.exists(db_path):
        import time
        time.sleep(0.5)  # Give Windows time to release file handles
        try:
            shutil.rmtree(db_path)
            print(f"  Deleted existing {encoder_name} database")
        except PermissionError:
            print(f"  ⚠️  Could not delete {encoder_name} database (files in use)")
            print(f"     Using existing database instead")
    
    print(f"Creating vector store with {encoder_name}...")
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=encoder_obj,
        persist_directory=db_path
    )
    
    # Get info
    collection = vector_store._collection
    sample = collection.get(limit=1, include=["embeddings"])
    
    # Check if embeddings exist and get dimensions
    try:
        if sample["embeddings"] is not None and len(sample["embeddings"]) > 0:
            dimensions = len(sample["embeddings"][0])
        else:
            dimensions = 0
    except (KeyError, TypeError, IndexError):
        dimensions = 0
    
    print(f"  ✅ Created {collection.count()} vectors with {dimensions} dimensions\n")
    
    return vector_store

# Create encoders
print("Setting up encoders...\n")

encoder_huggingface = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
encoder_openai_small = OpenAIEmbeddings(model="text-embedding-3-small")
encoder_openai_large = OpenAIEmbeddings(model="text-embedding-3-large")

# Create vector stores (this will cost a few cents for OpenAI embeddings)
print("Creating vector stores (OpenAI embeddings will incur small cost)...\n")

vs_hf = create_vector_store_with_encoder("huggingface", encoder_huggingface, chunks)
vs_small = create_vector_store_with_encoder("openai_small", encoder_openai_small, chunks)
vs_large = create_vector_store_with_encoder("openai_large", encoder_openai_large, chunks)

print("✅ All vector stores created!")


Setting up encoders...

Creating vector stores (OpenAI embeddings will incur small cost)...



Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  ⚠️  Could not delete huggingface database (files in use)
     Using existing database instead
Creating vector store with huggingface...
  ✅ Created 16 vectors with 384 dimensions

Creating vector store with openai_small...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  ✅ Created 8 vectors with 1536 dimensions

Creating vector store with openai_large...


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


  ✅ Created 8 vectors with 3072 dimensions

✅ All vector stores created!


### Visualize and Compare Encoders


In [20]:
# Function to visualize a vector store
def visualize_vector_store(vector_store, title):
    """Create 2D visualization of a vector store"""
    collection = vector_store._collection
    data = collection.get(include=["embeddings", "metadatas"])
    
    vectors = np.array(data["embeddings"])
    doc_types = [meta.get("doc_type", "unknown") for meta in data["metadatas"]]
    
    # Apply t-SNE
    print(f"Applying t-SNE for {title}...")
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
    vectors_2d = tsne.fit_transform(vectors)
    
    # Create plot
    fig = px.scatter(
        x=vectors_2d[:, 0],
        y=vectors_2d[:, 1],
        color=doc_types,
        title=title,
        labels={"x": "Dimension 1", "y": "Dimension 2", "color": "Document Type"},
        width=800,
        height=600
    )
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    
    return fig

# Visualize all three
print("\nGenerating visualizations...\n")

fig_hf = visualize_vector_store(vs_hf, "Hugging Face (all-MiniLM-L6-v2) - 384 dims")
fig_small = visualize_vector_store(vs_small, "OpenAI Small - 1,536 dims")
fig_large = visualize_vector_store(vs_large, "OpenAI Large - 3,072 dims")

print("\n✅ Visualizations ready!\n")

# Show them
print("Showing Hugging Face visualization:")
fig_hf.show()

print("\nShowing OpenAI Small visualization:")
fig_small.show()

print("\nShowing OpenAI Large visualization:")
fig_large.show()

print("\n📊 Compare the clustering quality:")
print("   - Hugging Face: Decent separation, some overlap")
print("   - OpenAI Small: Better separation, clearer clusters")
print("   - OpenAI Large: Best separation, very distinct clusters")


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given



Generating visualizations...

Applying t-SNE for Hugging Face (all-MiniLM-L6-v2) - 384 dims...


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Applying t-SNE for OpenAI Small - 1,536 dims...


Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Applying t-SNE for OpenAI Large - 3,072 dims...

✅ Visualizations ready!

Showing Hugging Face visualization:



Showing OpenAI Small visualization:



Showing OpenAI Large visualization:



📊 Compare the clustering quality:
   - Hugging Face: Decent separation, some overlap
   - OpenAI Small: Better separation, clearer clusters
   - OpenAI Large: Best separation, very distinct clusters


## Summary and Key Takeaways

Congratulations! You've built a complete RAG system from scratch. Here's what you accomplished:

### What You Built
1. ✅ Created a knowledge base with sample documents
2. ✅ Loaded documents with LangChain
3. ✅ Chunked documents with RecursiveCharacterTextSplitter
4. ✅ Created vector embeddings with encoder models
5. ✅ Stored vectors in Chroma database
6. ✅ Visualized vectors in 2D and 3D with t-SNE
7. ✅ Built a complete RAG pipeline
8. ✅ Created a Gradio chat interface
9. ✅ Compared different encoder models

### Key Concepts
- **Document Chunking**: Break large documents into focused pieces for better retrieval
- **Encoder Models**: Convert text to vectors (affects quality)
- **Vector Databases**: Store and search vectors efficiently (affects speed)
- **t-SNE**: Visualize high-dimensional data in 2D/3D
- **RAG Pipeline**: Retrieve → Augment → Generate
- **Conversation History**: LLMs are stateless, must pass full history

### Best Practices
1. **Start with open source encoders** (all-MiniLM-L6-v2) for development
2. **Upgrade to OpenAI embeddings** for production quality
3. **Experiment with chunk sizes** (typical: 500-2000 chars)
4. **Use chunk overlap** (10-20% of chunk size)
5. **Retrieve multiple chunks** (k=3-10)
6. **Handle conversation history** properly
7. **Visualize your vectors** to understand encoder behavior

### What Makes RAG Powerful
- ✅ **Handles typos** - Vector similarity is fuzzy
- ✅ **Understands synonyms** - "CEO" matches "Chief Executive Officer"
- ✅ **Works with private data** - Your documents, your knowledge
- ✅ **No retraining needed** - Just add documents and go
- ✅ **Scalable** - Works with millions of documents
- ✅ **Explainable** - Can show source documents

### Next Steps
1. **Add your own documents** - Replace sample data with real knowledge base
2. **Optimize chunking** - Experiment with different strategies
3. **Try different encoders** - Compare quality for your use case
4. **Add features** - Source citations, confidence scores, filters
5. **Deploy to production** - Separate ingestion from querying
6. **Measure performance** - Track retrieval quality and user satisfaction

### Resources
- LangChain: https://python.langchain.com/
- Chroma: https://www.trychroma.com/
- Sentence Transformers: https://www.sbert.net/
- OpenAI Embeddings: https://platform.openai.com/docs/guides/embeddings

---

**You now have a production-ready RAG system!** 🚀

Go forth and build amazing AI applications with retrieval-augmented generation!
